# RSNA Knee Abnormality Detection — Deep Multimodal EDA Phase 3B

## DINOv2-Small + Multilingual E5-Base — strict deep run

This notebook is the **replacement for the previous fallback Phase 3B notebook**.

It does **not** silently fall back to TF-IDF/SVD or PCA.

The goal is a controlled comparison:

- **Old text representation:** character TF-IDF + SVD
- **New text representation:** Multilingual E5-Base
- **Old MRI representation:** three 32×32 middle slices + PCA
- **New MRI representation:** DINOv2-Small features from sagittal, coronal, and axial representative slices

The notebook uses the exact CUDA-safe loading method that passed the Tesla T4 smoke tests:

```text
Multilingual E5-Base → AutoTokenizer + AutoModel
                      → attn_implementation="eager"

DINOv2-Small        → AutoImageProcessor + AutoModel
                      → attn_implementation="eager"
```

### Hard safety rule

If either model fails, the notebook should stop.  
There is **no automatic fallback** in this version.

### Expected successful modes

```text
TEXT_MODE  = multilingual_e5_base
IMAGE_MODE = dinov2_small

Text embedding shape  = (4407, 768)
Image embedding shape = (4407, 384)
```

In [ ]:
from pathlib import Path
from collections import Counter
import os, re, math, gc, warnings, unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom

import torch
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler, normalize
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import roc_auc_score, silhouette_score
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from sklearn.cross_decomposition import CCA
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)

SEED = 2026
rng = np.random.default_rng(SEED)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU capability:", torch.cuda.get_device_capability(0))
    print("Torch arch list:", torch.cuda.get_arch_list())

assert DEVICE == "cuda", "This deep EDA notebook is intended to run with a Kaggle GPU."

## 1. Configuration

These are the model paths that worked in your Kaggle setup.

If Kaggle changes the mounted paths in a future notebook version, edit only these two variables.

In [ ]:
TEXT_MODEL_PATH = "/kaggle/input/models/ranaabdulrehman145/intfloatmultilingual-e5-base/transformers/default/1"
IMAGE_MODEL_PATH = "/kaggle/input/models/metaresearch/dinov2/pytorch/small/1"

TEXT_BATCH_SIZE = 16
DEEP_IMAGE_BATCH_SIZE = 16

EXPECTED_N_STUDIES = 4407
EXPECTED_TEXT_DIM = 768
EXPECTED_IMAGE_DIM = 384

print("TEXT_MODEL_PATH :", TEXT_MODEL_PATH)
print("IMAGE_MODEL_PATH:", IMAGE_MODEL_PATH)

assert Path(TEXT_MODEL_PATH).exists(), f"E5 path not found: {TEXT_MODEL_PATH}"
assert Path(IMAGE_MODEL_PATH).exists(), f"DINOv2 path not found: {IMAGE_MODEL_PATH}"

print("✅ Both model paths exist")

## 2. CUDA and model smoke tests

This deliberately tests each model **separately** and frees GPU memory after each test.

If this cell fails, do not continue to the full run.

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoImageProcessor
from PIL import Image

# Basic CUDA test
x = torch.tensor([1.0, 2.0, 3.0], device="cuda")
y = x * 2
torch.cuda.synchronize()
print("CUDA tensor test:", y)
print("✅ Basic CUDA works")

# -----------------------------
# E5 smoke test
# -----------------------------
print("\nTesting Multilingual E5-Base...")

test_tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL_PATH,
    local_files_only=True
)

test_text_model = AutoModel.from_pretrained(
    TEXT_MODEL_PATH,
    local_files_only=True,
    attn_implementation="eager"
).to("cuda").eval()

sample_text = [
    "query: complete anterior cruciate ligament tear",
    "query: no meniscal tear"
]

sample_batch = test_tokenizer(
    sample_text,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)
sample_batch = {k: v.to("cuda") for k, v in sample_batch.items()}

with torch.inference_mode():
    out = test_text_model(**sample_batch)

mask = sample_batch["attention_mask"][..., None].bool()
sample_emb = out.last_hidden_state.masked_fill(~mask, 0.0).sum(dim=1)
sample_emb = sample_emb / sample_batch["attention_mask"].sum(dim=1)[..., None]
sample_emb = F.normalize(sample_emb, p=2, dim=1)

torch.cuda.synchronize()

print("E5 embedding shape:", tuple(sample_emb.shape))
assert sample_emb.shape == (2, EXPECTED_TEXT_DIM)
print("✅ E5 GPU smoke test successful")

del test_text_model, test_tokenizer, sample_batch, sample_emb, out
gc.collect()
torch.cuda.empty_cache()

# -----------------------------
# DINOv2 smoke test
# -----------------------------
print("\nTesting DINOv2-Small...")

test_processor = AutoImageProcessor.from_pretrained(
    IMAGE_MODEL_PATH,
    local_files_only=True
)

test_image_model = AutoModel.from_pretrained(
    IMAGE_MODEL_PATH,
    local_files_only=True,
    attn_implementation="eager"
).to("cuda").eval()

arr = np.random.default_rng(SEED).integers(
    0, 256, size=(224, 224), dtype=np.uint8
)
img = Image.fromarray(arr).convert("RGB")

inp = test_processor(images=[img], return_tensors="pt")
inp = {k: v.to("cuda") for k, v in inp.items()}

with torch.inference_mode():
    out = test_image_model(**inp)

sample_img_emb = out.last_hidden_state[:, 0]
sample_img_emb = F.normalize(sample_img_emb, dim=1)

torch.cuda.synchronize()

print("DINOv2 embedding shape:", tuple(sample_img_emb.shape))
assert sample_img_emb.shape == (1, EXPECTED_IMAGE_DIM)
print("✅ DINOv2 GPU smoke test successful")

del test_image_model, test_processor, inp, out, sample_img_emb
gc.collect()
torch.cuda.empty_cache()

print("\n✅ ALL PREFLIGHT TESTS PASSED")

## 3. Load the RSNA competition data

In [ ]:
CANDIDATES = [
    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
    Path("/kaggle/input/rsna-knee-abnormality-detection"),
]

DATA_DIR = next((p for p in CANDIDATES if p.exists()), None)

if DATA_DIR is None:
    matches = list(Path("/kaggle/input").rglob("rsna-knee-abnormality-detection"))
    DATA_DIR = matches[0] if matches else None

if DATA_DIR is None:
    raise FileNotFoundError("RSNA competition data not found.")

train = pd.read_csv(DATA_DIR / "train.csv")
train_series = pd.read_csv(DATA_DIR / "train_series.csv")

train["is_gold"] = train[TARGETS].notna().all(axis=1)

print("Competition root:", DATA_DIR)
print("Studies:", len(train))
print("Series:", len(train_series))
print("Gold:", int(train["is_gold"].sum()))

assert len(train) == EXPECTED_N_STUDIES
assert int(train["is_gold"].sum()) == 58

# 4. Deep text embeddings — Multilingual E5-Base

We use full radiology reports with the `query:` prefix.

The model is loaded with:

```python
attn_implementation="eager"
```

because this is the CUDA-safe configuration verified on the Tesla T4.

### Important limitation

E5-Base has a finite token context. Reports longer than 512 tokens are truncated in this controlled experiment.

We therefore audit the token-length distribution before encoding.

In [ ]:
reports = train["Report"].fillna("").astype(str).tolist()

tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL_PATH,
    local_files_only=True
)

# Token length audit before truncation
token_lengths = []
for start in range(0, len(reports), 128):
    batch_text = ["query: " + x for x in reports[start:start+128]]
    toks = tokenizer(
        batch_text,
        padding=False,
        truncation=False,
        add_special_tokens=True
    )
    token_lengths.extend([len(ids) for ids in toks["input_ids"]])

token_lengths = np.asarray(token_lengths)

print("Report token-length quantiles:")
display(
    pd.Series(token_lengths).quantile(
        [0, .25, .50, .75, .90, .95, .99, 1]
    ).to_frame("tokens")
)
print("Reports >512 tokens:", int((token_lengths > 512).sum()),
      "/", len(token_lengths),
      f"({100*(token_lengths > 512).mean():.2f}%)")

In [ ]:
print("Loading Multilingual E5-Base for full extraction...")

text_model = AutoModel.from_pretrained(
    TEXT_MODEL_PATH,
    local_files_only=True,
    attn_implementation="eager"
).to(DEVICE).eval()

text_inputs = ["query: " + r for r in reports]
text_embeddings = []

with torch.inference_mode():
    for start in range(0, len(text_inputs), TEXT_BATCH_SIZE):
        batch_text = text_inputs[start:start + TEXT_BATCH_SIZE]

        batch = tokenizer(
            batch_text,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        outputs = text_model(**batch)

        mask = batch["attention_mask"][..., None].bool()
        emb = outputs.last_hidden_state.masked_fill(~mask, 0.0).sum(dim=1)
        emb = emb / batch["attention_mask"].sum(dim=1)[..., None]
        emb = F.normalize(emb, p=2, dim=1)

        text_embeddings.append(
            emb.cpu().numpy().astype(np.float32)
        )

        done = min(start + TEXT_BATCH_SIZE, len(text_inputs))
        if done % 400 < TEXT_BATCH_SIZE or done == len(text_inputs):
            print(f"E5 reports: {done}/{len(text_inputs)}")

X_text = np.concatenate(text_embeddings, axis=0)
TEXT_MODE = "multilingual_e5_base"

print("TEXT_MODE:", TEXT_MODE)
print("Text embedding shape:", X_text.shape)

assert X_text.shape == (EXPECTED_N_STUDIES, EXPECTED_TEXT_DIM)
assert np.isfinite(X_text).all()
assert TEXT_MODE == "multilingual_e5_base"

np.save("/kaggle/working/rsna_text_embeddings_deep.npy", X_text)
print("✅ E5 embeddings verified and cached")

# Free GPU memory before loading DINOv2
del text_model, text_embeddings
gc.collect()
torch.cuda.empty_cache()

# 5. Select one representative MRI series per anatomical plane

Policy for this controlled EDA:

1. prefer fluid-sensitive series;
2. if multiple candidates exist in a plane, prefer the one with the largest slice count.

If the Advanced EDA inventory is attached as an input, this notebook reuses its `n_files`.
Otherwise it counts files directly.

In [ ]:
# Optional acceleration from the Advanced EDA output
adv_matches = list(Path("/kaggle/input").rglob("rsna_advanced_series_inventory_train.csv"))
ADV_INV_PATH = adv_matches[0] if adv_matches else None

sel = train_series.copy()

if ADV_INV_PATH is not None:
    print("Using Advanced EDA inventory:", ADV_INV_PATH)
    adv = pd.read_csv(ADV_INV_PATH)

    nfile_map = (
        adv[["StudyInstanceUID", "SeriesInstanceUID", "n_files"]]
        .drop_duplicates(["StudyInstanceUID", "SeriesInstanceUID"])
    )

    sel = sel.merge(
        nfile_map,
        on=["StudyInstanceUID", "SeriesInstanceUID"],
        how="left"
    )
else:
    print("Advanced inventory not attached — counting files directly.")

    def count_files(study, series):
        folder = DATA_DIR / "train_series" / str(study) / str(series)
        try:
            return sum(1 for p in folder.iterdir() if p.is_file())
        except Exception:
            return 0

    sel["n_files"] = [
        count_files(s, u)
        for s, u in zip(sel["StudyInstanceUID"], sel["SeriesInstanceUID"])
    ]

# Safety fallback only for missing counts, not model embeddings
missing_n = sel["n_files"].isna()
if missing_n.any():
    print("Filling", int(missing_n.sum()), "missing series counts from disk.")
    for idx in sel.index[missing_n]:
        r = sel.loc[idx]
        folder = DATA_DIR / "train_series" / str(r["StudyInstanceUID"]) / str(r["SeriesInstanceUID"])
        try:
            sel.loc[idx, "n_files"] = sum(1 for p in folder.iterdir() if p.is_file())
        except Exception:
            sel.loc[idx, "n_files"] = 0

rows = []

for study, g in sel.groupby("StudyInstanceUID"):
    for plane in ["Sagittal", "Coronal", "Axial"]:
        p = g[g["Anatomical_Plane"].eq(plane)].copy()

        if len(p) == 0:
            continue

        p = p.sort_values(
            ["Fluid_Sensitive", "n_files"],
            ascending=[False, False]
        )

        r = p.iloc[0]

        rows.append({
            "StudyInstanceUID": str(study),
            "Plane": plane,
            "SeriesInstanceUID": str(r["SeriesInstanceUID"]),
            "Fluid_Sensitive": int(r["Fluid_Sensitive"]),
            "n_files": int(r["n_files"])
        })

selected = pd.DataFrame(rows)

display(selected.head(10))
display(pd.crosstab(selected["Plane"], selected["Fluid_Sensitive"]))

selected.to_csv(
    "/kaggle/working/rsna_selected_series_deep.csv",
    index=False
)

## 6. DICOM spatial ordering and middle-slice loader

In [ ]:
def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def spatial_coordinate(ds):
    try:
        ori = np.asarray([float(x) for x in ds.ImageOrientationPatient])
        pos = np.asarray([float(x) for x in ds.ImagePositionPatient])
        normal = np.cross(ori[:3], ori[3:])
        return float(np.dot(pos, normal))
    except Exception:
        return np.nan

def order_key(path):
    try:
        ds = pydicom.dcmread(str(path), stop_before_pixels=True)

        z = spatial_coordinate(ds)
        if np.isfinite(z):
            return (0, z)

        inst = safe_float(getattr(ds, "InstanceNumber", np.nan))
        if np.isfinite(inst):
            return (1, inst)

    except Exception:
        pass

    return (2, path.name)

def read_middle_slice(study, series):
    folder = DATA_DIR / "train_series" / str(study) / str(series)
    files = [p for p in folder.iterdir() if p.is_file()]

    if not files:
        return None

    files = sorted(files, key=order_key)
    f = files[len(files) // 2]

    ds = pydicom.dcmread(str(f))
    img = ds.pixel_array.astype(np.float32)

    slope = safe_float(getattr(ds, "RescaleSlope", 1.0))
    intercept = safe_float(getattr(ds, "RescaleIntercept", 0.0))

    if not np.isfinite(slope):
        slope = 1.0
    if not np.isfinite(intercept):
        intercept = 0.0

    img = img * slope + intercept

    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        img = img.max() - img

    lo, hi = np.percentile(img, [1, 99])

    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(img, dtype=np.float32)

    img = np.clip(img, lo, hi)
    img = (img - lo) / (hi - lo)

    return img.astype(np.float32)

# 7. Deep MRI embeddings — DINOv2-Small

For each study:

```text
fluid-sensitive sagittal representative slice
+
fluid-sensitive coronal representative slice
+
fluid-sensitive axial representative slice
        ↓
DINOv2-Small
        ↓
three 384-D embeddings
        ↓
mean + L2 normalization
        ↓
one 384-D study embedding
```

This is still an EDA representation, not the final classifier architecture.

In [ ]:
study_ids = train["StudyInstanceUID"].astype(str).tolist()
plane_order = ["Sagittal", "Coronal", "Axial"]

selection_map = {
    (str(r.StudyInstanceUID), r.Plane): str(r.SeriesInstanceUID)
    for r in selected.itertuples(index=False)
}

valid_study = [
    study for study in study_ids
    if all((study, plane) in selection_map for plane in plane_order)
]

print("Studies with all 3 selected planes:", len(valid_study))
assert len(valid_study) == EXPECTED_N_STUDIES, (
    f"Expected {EXPECTED_N_STUDIES} complete studies, got {len(valid_study)}"
)

In [ ]:
print("Loading DINOv2-Small for full extraction...")

processor = AutoImageProcessor.from_pretrained(
    IMAGE_MODEL_PATH,
    local_files_only=True
)

image_model = AutoModel.from_pretrained(
    IMAGE_MODEL_PATH,
    local_files_only=True,
    attn_implementation="eager"
).to(DEVICE).eval()

deep_features = []

with torch.inference_mode():
    for start in range(0, len(valid_study), DEEP_IMAGE_BATCH_SIZE):
        batch_ids = valid_study[start:start + DEEP_IMAGE_BATCH_SIZE]

        plane_embeddings = []

        for plane in plane_order:
            pil_images = []

            for study in batch_ids:
                series = selection_map[(study, plane)]
                img = read_middle_slice(study, series)

                if img is None:
                    raise RuntimeError(
                        f"Could not load representative slice: study={study}, plane={plane}"
                    )

                arr = (np.clip(img, 0, 1) * 255).astype(np.uint8)
                pil_images.append(
                    Image.fromarray(arr).convert("RGB")
                )

            inp = processor(
                images=pil_images,
                return_tensors="pt"
            )
            inp = {k: v.to(DEVICE) for k, v in inp.items()}

            out = image_model(**inp)

            # DINOv2 CLS token
            z = out.last_hidden_state[:, 0]
            z = F.normalize(z, dim=1)

            plane_embeddings.append(
                z.cpu().numpy().astype(np.float32)
            )

        study_z = np.mean(
            np.stack(plane_embeddings, axis=1),
            axis=1
        )
        study_z = normalize(study_z).astype(np.float32)

        deep_features.append(study_z)

        done = min(start + DEEP_IMAGE_BATCH_SIZE, len(valid_study))
        print(f"DINOv2 studies: {done}/{len(valid_study)}")

X_image = np.concatenate(deep_features, axis=0)
image_study_ids = valid_study.copy()
IMAGE_MODE = "dinov2_small"

print("IMAGE_MODE:", IMAGE_MODE)
print("Image embedding shape:", X_image.shape)

assert X_image.shape == (EXPECTED_N_STUDIES, EXPECTED_IMAGE_DIM)
assert np.isfinite(X_image).all()
assert IMAGE_MODE == "dinov2_small"

np.save(
    "/kaggle/working/rsna_image_embeddings_deep.npy",
    X_image
)

print("✅ DINOv2 embeddings verified and cached")

del image_model, processor, deep_features
gc.collect()
torch.cuda.empty_cache()

## 8. Align MRI and report embeddings

In [ ]:
uid_to_train_idx = {
    str(uid): i
    for i, uid in enumerate(train["StudyInstanceUID"].astype(str))
}

aligned_idx = [uid_to_train_idx[u] for u in image_study_ids]

X_text_a = X_text[aligned_idx]
aligned = train.iloc[aligned_idx].reset_index(drop=True).copy()

assert len(aligned) == len(X_image) == len(X_text_a)
assert X_text_a.shape[1] == EXPECTED_TEXT_DIM
assert X_image.shape[1] == EXPECTED_IMAGE_DIM

aligned["report_token_length"] = token_lengths[aligned_idx]

print("Aligned paired studies:", len(aligned))
print("Text shape:", X_text_a.shape)
print("Image shape:", X_image.shape)

# 9. Unsupervised 2-D maps

UMAP is used when installed; PCA is the fallback **only for visualization**, not for the deep feature representation itself.

In [ ]:
try:
    import umap

    img_2d = umap.UMAP(
        n_neighbors=30,
        min_dist=0.1,
        metric="cosine",
        random_state=SEED
    ).fit_transform(X_image)

    txt_2d = umap.UMAP(
        n_neighbors=30,
        min_dist=0.1,
        metric="cosine",
        random_state=SEED
    ).fit_transform(X_text_a)

    MAP_METHOD = "UMAP"

except Exception as e:
    print("UMAP unavailable; using PCA only for visualization:",
          type(e).__name__)

    img_2d = PCA(
        n_components=2,
        random_state=SEED
    ).fit_transform(X_image)

    txt_2d = PCA(
        n_components=2,
        random_state=SEED
    ).fit_transform(X_text_a)

    MAP_METHOD = "PCA"

print("Map method:", MAP_METHOD)

## 10. Rough language grouping for nuisance analysis

In [ ]:
def strip_accents(s):
    s = unicodedata.normalize("NFKD", str(s))
    return "".join(ch for ch in s if not unicodedata.combining(ch))

LANG_HINTS = {
    "English": [
        " the ", " knee ", " ligament ", " meniscus ",
        " findings ", " impression "
    ],
    "Spanish": [
        " rodilla ", " menisco ", " hallazgos ",
        " impresion ", " rotura ", " derrame "
    ],
    "French": [
        " genou ", " menisque ", " constatations ",
        " aucune ", " sans "
    ],
    "German/Dutch": [
        " knie ", " menisk", " kreuzband",
        " geen ", " keine ", " ohne "
    ],
    "Turkish": [
        " diz ", " menisk", " bulgu",
        " izlen", " yoktur ", " eklem "
    ],
    "Balkan-Latin": [
        " koljen", " kolen", " menisk",
        " ligament", " nema ", " nalaz"
    ],
}

def rough_lang(text):
    text = str(text)

    greek = sum("\u0370" <= c <= "\u03ff" for c in text)
    cyr = sum("\u0400" <= c <= "\u04ff" for c in text)

    if greek > 10:
        return "Greek"
    if cyr > 10:
        return "Cyrillic"

    t = " " + strip_accents(text).casefold() + " "

    scores = {
        k: sum(
            t.count(strip_accents(v).casefold())
            for v in vals
        )
        for k, vals in LANG_HINTS.items()
    }

    best = max(scores, key=scores.get)

    return best if scores[best] > 0 else "Latin-Other"

aligned["language"] = aligned["Report"].map(rough_lang)

display(aligned["language"].value_counts().to_frame("studies"))

## 11. Scanner manufacturer and field-strength extraction

In [ ]:
manufacturers = []
fields = []

for study in image_study_ids:
    series = selection_map[(study, "Sagittal")]
    folder = DATA_DIR / "train_series" / study / series
    files = [p for p in folder.iterdir() if p.is_file()]

    try:
        ds = pydicom.dcmread(
            str(files[0]),
            stop_before_pixels=True
        )
        manufacturers.append(
            str(getattr(ds, "Manufacturer", "")).strip()
        )
        fields.append(
            safe_float(getattr(ds, "MagneticFieldStrength", np.nan))
        )
    except Exception:
        manufacturers.append("")
        fields.append(np.nan)

aligned["manufacturer"] = manufacturers
aligned["field_strength"] = fields

display(aligned["manufacturer"].value_counts().to_frame("studies"))
display(aligned["field_strength"].value_counts(dropna=False).sort_index().to_frame("studies"))

## 12. Deep image embedding map by manufacturer

In [ ]:
plt.figure(figsize=(9, 7))

for vendor in aligned["manufacturer"].value_counts().head(12).index:
    m = aligned["manufacturer"].eq(vendor)
    plt.scatter(
        img_2d[m, 0],
        img_2d[m, 1],
        s=8,
        alpha=0.6,
        label=vendor or "missing"
    )

plt.xlabel(f"{MAP_METHOD} 1")
plt.ylabel(f"{MAP_METHOD} 2")
plt.title(f"Image embedding space ({IMAGE_MODE}) by manufacturer")
plt.legend(markerscale=2)
plt.show()

## 13. Deep text embedding map by language

In [ ]:
plt.figure(figsize=(9, 7))

for lang in aligned["language"].value_counts().index:
    m = aligned["language"].eq(lang)
    plt.scatter(
        txt_2d[m, 0],
        txt_2d[m, 1],
        s=8,
        alpha=0.6,
        label=lang
    )

plt.xlabel(f"{MAP_METHOD} 1")
plt.ylabel(f"{MAP_METHOD} 2")
plt.title(f"Text embedding space ({TEXT_MODE}) by language")
plt.legend(markerscale=2)
plt.show()

# 14. Domain dominance

We reproduce the same nuisance tests as the fallback experiment:

- MRI embedding → scanner manufacturer
- text embedding → language

This gives us a directly comparable diagnostic.

In [ ]:
def common_classes(y, min_count=20):
    s = pd.Series(y).fillna("MISSING").astype(str)
    keep = set(
        s.value_counts()[lambda x: x >= min_count].index
    )
    return s.where(s.isin(keep), "OTHER").values

vendor_y = common_classes(
    aligned["manufacturer"], 20
)
lang_y = common_classes(
    aligned["language"], 20
)

domain_clf = LogisticRegression(
    max_iter=2500,
    class_weight="balanced",
    solver="liblinear",
    multi_class="ovr"
)

vendor_acc = cross_val_score(
    domain_clf,
    X_image,
    vendor_y,
    cv=StratifiedKFold(
        5,
        shuffle=True,
        random_state=SEED
    ),
    scoring="balanced_accuracy"
)

lang_acc = cross_val_score(
    domain_clf,
    X_text_a,
    lang_y,
    cv=StratifiedKFold(
        5,
        shuffle=True,
        random_state=SEED
    ),
    scoring="balanced_accuracy"
)

print(
    "Image → manufacturer balanced accuracy:",
    vendor_acc.mean(),
    "+/-",
    vendor_acc.std()
)

print(
    "Text → language balanced accuracy:",
    lang_acc.mean(),
    "+/-",
    lang_acc.std()
)

# 15. MRI/report nearest-neighbor agreement

In [ ]:
K_NEIGH = 10

img_nn = NearestNeighbors(
    n_neighbors=K_NEIGH + 1,
    metric="cosine",
    n_jobs=-1
).fit(X_image)

txt_nn = NearestNeighbors(
    n_neighbors=K_NEIGH + 1,
    metric="cosine",
    n_jobs=-1
).fit(X_text_a)

_, img_idx = img_nn.kneighbors(X_image)
_, txt_idx = txt_nn.kneighbors(X_text_a)

overlaps = []

for i in range(len(aligned)):
    a = set(img_idx[i, 1:])
    b = set(txt_idx[i, 1:])
    overlaps.append(
        len(a & b) / len(a | b)
    )

overlaps = np.asarray(overlaps)

perm = rng.permutation(len(aligned))
rand = []

for i in range(len(aligned)):
    a = set(img_idx[i, 1:])
    b = set(txt_idx[perm[i], 1:])
    rand.append(
        len(a & b) / len(a | b)
    )

rand = np.asarray(rand)

print(
    "Mean image/text neighbor Jaccard:",
    overlaps.mean()
)
print(
    "Randomized baseline:",
    rand.mean()
)
print(
    "Difference:",
    overlaps.mean() - rand.mean()
)

display(
    pd.Series(overlaps).describe().to_frame(
        "neighbor_jaccard"
    )
)

# 16. Cross-validated MRI↔report alignment with CCA

We reproduce the same cross-validated matched-vs-permuted test used in the fallback run.

In [ ]:
NX = min(32, X_image.shape[1], len(X_image) - 1)
NY = min(32, X_text_a.shape[1], len(X_text_a) - 1)

Xi = PCA(
    n_components=NX,
    random_state=SEED
).fit_transform(X_image)

Xt = PCA(
    n_components=NY,
    random_state=SEED
).fit_transform(X_text_a)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

matched_scores = []
perm_scores = []

for fold, (tr, va) in enumerate(kf.split(Xi)):
    sx = StandardScaler().fit(Xi[tr])
    sy = StandardScaler().fit(Xt[tr])

    xtr = sx.transform(Xi[tr])
    xva = sx.transform(Xi[va])

    ytr = sy.transform(Xt[tr])
    yva = sy.transform(Xt[va])

    ncca = min(
        12,
        xtr.shape[1],
        ytr.shape[1]
    )

    cca = CCA(
        n_components=ncca,
        max_iter=1000
    )

    cca.fit(xtr, ytr)

    zx, zy = cca.transform(xva, yva)

    zx = normalize(zx)
    zy = normalize(zy)

    matched = np.sum(zx * zy, axis=1)

    p = rng.permutation(len(va))
    mismatched = np.sum(
        zx * zy[p],
        axis=1
    )

    matched_scores.extend(
        matched.tolist()
    )
    perm_scores.extend(
        mismatched.tolist()
    )

    print(
        "fold",
        fold,
        "matched",
        float(np.mean(matched)),
        "permuted",
        float(np.mean(mismatched))
    )

matched_scores = np.asarray(matched_scores)
perm_scores = np.asarray(perm_scores)

print(
    "\nOOF matched cosine mean:",
    matched_scores.mean()
)
print(
    "OOF permuted cosine mean:",
    perm_scores.mean()
)
print(
    "Separation:",
    matched_scores.mean() - perm_scores.mean()
)

# 17. Deep visual outliers and near duplicates

In [ ]:
knn_img = NearestNeighbors(
    n_neighbors=11,
    metric="cosine",
    n_jobs=-1
).fit(X_image)

d_img, j_img = knn_img.kneighbors(X_image)

img_issue = aligned[[
    "StudyInstanceUID",
    "manufacturer",
    "field_strength",
    "language"
]].copy()

img_issue["nearest_visual_distance"] = d_img[:, 1]
img_issue["mean_10nn_visual_distance"] = d_img[:, 1:].mean(axis=1)
img_issue["nearest_visual_uid"] = (
    aligned.iloc[j_img[:, 1]]["StudyInstanceUID"].values
)

print("Visual outliers:")
display(
    img_issue.sort_values(
        "mean_10nn_visual_distance",
        ascending=False
    ).head(30)
)

print("\nVisual near duplicates:")
display(
    img_issue.sort_values(
        "nearest_visual_distance"
    ).head(30)
)

# 18. Exploratory gold-set kNN signal

This is still a **representation diagnostic**, not a trustworthy competition CV estimate.

Only 58 studies have gold target vectors.

In [ ]:
aligned_gold = aligned[
    aligned["is_gold"]
].copy()

gold_positions = np.where(
    aligned["is_gold"].values
)[0]

Yg = aligned_gold[TARGETS].astype(int).values

def gold_knn_auc(X, k=7):
    Xg = X[gold_positions]

    nn = NearestNeighbors(
        n_neighbors=min(k + 1, len(Xg)),
        metric="cosine"
    ).fit(Xg)

    d, j = nn.kneighbors(Xg)

    pred = []

    for i in range(len(Xg)):
        neigh = j[i, 1:]
        w = 1 / (d[i, 1:] + 1e-4)
        w = w / w.sum()

        pred.append(
            (Yg[neigh] * w[:, None]).sum(axis=0)
        )

    pred = np.asarray(pred)

    aucs = {}

    for t, target in enumerate(TARGETS):
        if len(np.unique(Yg[:, t])) < 2:
            aucs[target] = np.nan
        else:
            aucs[target] = roc_auc_score(
                Yg[:, t],
                pred[:, t]
            )

    return aucs, pred

img_auc, _ = gold_knn_auc(X_image)
txt_auc, _ = gold_knn_auc(X_text_a)

# Controlled naive combined embedding for direct comparison
zi = normalize(
    PCA(
        n_components=min(64, X_image.shape[1]),
        random_state=SEED
    ).fit_transform(X_image)
)

zt = normalize(
    PCA(
        n_components=min(64, X_text_a.shape[1]),
        random_state=SEED
    ).fit_transform(X_text_a)
)

X_combo = normalize(
    np.concatenate([zi, zt], axis=1)
)

combo_auc, _ = gold_knn_auc(X_combo)

auc_table = pd.DataFrame({
    "Image": img_auc,
    "Text": txt_auc,
    "Combined": combo_auc
})

display(auc_table)

print("Exploratory macro AUC:")
display(
    auc_table.mean().to_frame("macro")
)

# 19. Deep multimodal cluster discovery

In [ ]:
K_VALUES = [8, 12, 16, 24]
scores = []

sample = np.arange(len(X_combo))

if len(sample) > 2500:
    sample = rng.choice(
        sample,
        2500,
        replace=False
    )

for k in K_VALUES:
    km = MiniBatchKMeans(
        n_clusters=k,
        random_state=SEED,
        n_init="auto",
        batch_size=512
    )

    labels = km.fit_predict(X_combo)

    try:
        sil = silhouette_score(
            X_combo[sample],
            labels[sample]
        )
    except Exception:
        sil = np.nan

    scores.append((k, sil))

cluster_score_table = pd.DataFrame(
    scores,
    columns=["k", "silhouette"]
)

display(cluster_score_table)

good = [
    x for x in scores
    if np.isfinite(x[1])
]

BEST_K = (
    max(good, key=lambda x: x[1])[0]
    if good else 12
)

multimodal_km = MiniBatchKMeans(
    n_clusters=BEST_K,
    random_state=SEED,
    n_init="auto",
    batch_size=512
)

aligned["multimodal_cluster"] = (
    multimodal_km.fit_predict(X_combo)
)

print("Best K:", BEST_K)

print("\nClusters × language")
display(
    pd.crosstab(
        aligned["multimodal_cluster"],
        aligned["language"]
    )
)

print("\nClusters × manufacturer")
display(
    pd.crosstab(
        aligned["multimodal_cluster"],
        aligned["manufacturer"]
    )
)

## 20. Gold-label enrichment by deep multimodal cluster

In [ ]:
cluster_gold = aligned[
    aligned["is_gold"]
].copy()

for target in TARGETS:
    tab = (
        cluster_gold
        .groupby("multimodal_cluster")[target]
        .agg(["count", "mean", "sum"])
        .rename(columns={
            "mean": "positive_rate",
            "sum": "positives"
        })
    )

    print("\n", target)
    display(tab)

# 21. Optional comparison to a previously attached fallback Phase 3B summary

If you attach the old fallback notebook output to this Kaggle notebook, this cell tries to find its summary CSV automatically.

This comparison is optional.

In [ ]:
fallback_candidates = [
    p for p in Path("/kaggle/input").rglob("rsna_multimodal_eda_summary.csv")
]

if fallback_candidates:
    fallback_path = fallback_candidates[0]
    print("Fallback summary found:", fallback_path)

    fallback_summary = pd.read_csv(fallback_path)
    display(fallback_summary)
else:
    fallback_path = None
    print(
        "No fallback summary attached. "
        "That is okay — the deep outputs will still be saved."
    )

# 22. Save deep artifacts

All names use the `_deep` suffix so the fallback baseline is not overwritten.

In [ ]:
OUT = Path("/kaggle/working")

np.save(
    OUT / "rsna_text_embeddings_deep.npy",
    X_text_a
)

np.save(
    OUT / "rsna_image_embeddings_deep.npy",
    X_image
)

np.save(
    OUT / "rsna_combined_embeddings_deep.npy",
    X_combo
)

aligned[[
    "StudyInstanceUID",
    "is_gold",
    "language",
    "manufacturer",
    "field_strength",
    "report_token_length",
    "multimodal_cluster"
]].to_csv(
    OUT / "rsna_multimodal_embedding_index_deep.csv",
    index=False
)

img_issue.to_csv(
    OUT / "rsna_visual_embedding_issues_deep.csv",
    index=False
)

auc_table.to_csv(
    OUT / "rsna_gold_embedding_knn_auc_deep.csv"
)

cluster_score_table.to_csv(
    OUT / "rsna_multimodal_cluster_scores_deep.csv",
    index=False
)

summary = pd.DataFrame([
    {
        "metric": "text_mode",
        "value": TEXT_MODE
    },
    {
        "metric": "image_mode",
        "value": IMAGE_MODE
    },
    {
        "metric": "paired_studies",
        "value": len(aligned)
    },
    {
        "metric": "text_embedding_dim",
        "value": X_text_a.shape[1]
    },
    {
        "metric": "image_embedding_dim",
        "value": X_image.shape[1]
    },
    {
        "metric": "reports_over_512_tokens",
        "value": int((aligned["report_token_length"] > 512).sum())
    },
    {
        "metric": "image_to_manufacturer_balanced_accuracy_mean",
        "value": vendor_acc.mean()
    },
    {
        "metric": "image_to_manufacturer_balanced_accuracy_std",
        "value": vendor_acc.std()
    },
    {
        "metric": "text_to_language_balanced_accuracy_mean",
        "value": lang_acc.mean()
    },
    {
        "metric": "text_to_language_balanced_accuracy_std",
        "value": lang_acc.std()
    },
    {
        "metric": "neighbor_jaccard_mean",
        "value": overlaps.mean()
    },
    {
        "metric": "neighbor_jaccard_random",
        "value": rand.mean()
    },
    {
        "metric": "cca_matched_mean",
        "value": matched_scores.mean()
    },
    {
        "metric": "cca_permuted_mean",
        "value": perm_scores.mean()
    },
    {
        "metric": "cca_separation",
        "value": matched_scores.mean() - perm_scores.mean()
    },
    {
        "metric": "image_knn_macro_auc_gold",
        "value": auc_table["Image"].mean()
    },
    {
        "metric": "text_knn_macro_auc_gold",
        "value": auc_table["Text"].mean()
    },
    {
        "metric": "combined_knn_macro_auc_gold",
        "value": auc_table["Combined"].mean()
    },
    {
        "metric": "best_multimodal_cluster_k",
        "value": BEST_K
    }
])

summary.to_csv(
    OUT / "rsna_multimodal_eda_summary_deep.csv",
    index=False
)

display(summary)

print("\nSaved deep artifacts:")
for p in sorted(OUT.glob("rsna_*_deep.*")):
    print(" -", p.name)

# 23. Files to send back

After this notebook finishes, send these files:

```text
rsna_multimodal_eda_summary_deep.csv
rsna_gold_embedding_knn_auc_deep.csv
rsna_multimodal_embedding_index_deep.csv
rsna_visual_embedding_issues_deep.csv
rsna_multimodal_cluster_scores_deep.csv

rsna_text_embeddings_deep.npy
rsna_image_embeddings_deep.npy
rsna_combined_embeddings_deep.npy

the executed notebook
```

The first things we will compare against the fallback run are:

```text
MRI pathology kNN AUC
Text pathology kNN AUC
Combined kNN AUC

MRI → scanner manufacturer
Text → language

MRI/report neighbor overlap
CCA matched vs randomized alignment
```

Only after that comparison do we decide what representation should move into Phase 3C.